## Getting Started with AgentCore, Strands Agents and A2A

[A2A protocol](https://a2a-protocol.org/dev/specification/) is an open standard designed to facilitate communication and interoperability between independent, potentially opaque AI agent systems. In an ecosystem where agents might be built using different frameworks, languages, or by different vendors, A2A provides a common language and interaction model.

[Amazon AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) provides a secure, serverless and purpose-built hosting environment for deploying and running AI agents or tools. 

Recently, AWS announced [A2A support](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-a2a.html) for AgentCore Runtime.

In this workshop, you are going to build following architecture, using AgentCore Runtime:

<img src="images/architecture-getting-started.png" style="width: 80%;">

In this getting started notebook, we are going to build two agents. First agent is an AWS Docs expert. It will query AWS Docs MCP to read and search AWS Documentation and also generate recommendations. Second agent is an AWS Blog expert. It will use websearch to look into AWS latest blogs and news.

So let's get started!

### Setup

Install dependencies

In [1]:
%uv pip install -q -r requirements.txt --no-cache-dir --force-reinstall

Note: you may need to restart the kernel to use updated packages.


**Please restart your environment, so it can reflect new versions!**

In [ ]:
#import IPython

#IPython.Application.instance().kernel.do_shutdown(True)

Checking if `bedrock-agentcore-starter-toolkit` version is 0.1.21

In [2]:
!uv pip freeze | grep boto
!uv pip freeze | grep agentcore

Using Python 3.14.3 environment at: /Users/eric.fu/projects/xealth/agentcore-samples/.venv
boto3==1.40.50
botocore==1.40.76
Using Python 3.14.3 environment at: /Users/eric.fu/projects/xealth/agentcore-samples/.venv
bedrock-agentcore==0.1.7
bedrock-agentcore-starter-toolkit==0.1.24


In [3]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
user_name = os.getenv("USER_NAME")
if not user_name:
    raise ValueError("USER_NAME environment variable is not set. Please set it in the .env file.")


In [4]:
# Import libraries
import os
import json
import requests
import boto3
import time
from boto3.session import Session
from strands.tools import tool

# Get boto session
boto_session = Session()

### 1 - Create Code for the two agents

Create `agents` folder if it's not created.

In [5]:
![ ! -d "agents" ] && mkdir agents

#### 1.1 - AWS Docs expert Agent

Firstly let's write our first agent code to a file locally; this agent will later be deployed to AgentCore runtime.

In [6]:
%%writefile agents/strands_aws_docs.py
import os
import logging
import asyncio
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.multiagent.a2a import A2AServer
from strands.tools.mcp import MCPClient
from fastapi import FastAPI
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI()
runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')
host, port = "0.0.0.0", 9000

# Global MCP client with lazy initialization
_mcp_client = None

async def get_mcp_client():
    """Lazy initialization of MCP client with timeout"""
    global _mcp_client
    if _mcp_client is None:
        try:
            _mcp_client = MCPClient(
                lambda: stdio_client(
                    StdioServerParameters(
                        command="uvx",
                        args=["awslabs.aws-documentation-mcp-server@latest"]
                    )
                )
            )
            # Start with timeout
            await asyncio.wait_for(_mcp_client.start(), timeout=10.0)
            logger.info("MCP client initialized")
        except asyncio.TimeoutError:
            logger.error("MCP client startup timed out")
            _mcp_client = None
        except Exception as e:
            logger.error(f"MCP client failed: {e}")
            _mcp_client = None
    return _mcp_client

system_prompt = """You are an AWS Documentation Assistant powered by the AWS Documentation MCP server. Your role is to help users find accurate, up-to-date information from AWS documentation.

CRITICAL: Keep responses SHORT and FOCUSED.

Guidelines:
- Provide concise, actionable answers (max 3 sentences)
- Use bullet points for lists
- Skip verbose explanations
- If MCP is unavailable, provide basic AWS knowledge
- Timeout operations after 8 seconds
- Prioritize speed over completeness

You have access to AWS documentation search tools when available."""

# Initialize agent with minimal tools first
agent = Agent(
    system_prompt=system_prompt,
    tools=[],  # Start with no tools, add dynamically
    name="AWS Docs Agent",
    description="An agent to query AWS Docs using AWS MCP.",
)

# Add tools dynamically when MCP is ready
async def setup_agent_tools():
    """Setup agent tools when MCP client is ready"""
    try:
        mcp_client = await get_mcp_client()
        if mcp_client:
            tools = await asyncio.wait_for(
                mcp_client.list_tools_async(),
                timeout=5.0
            )
            agent.tools = [tools] if tools else []
            logger.info("Agent tools configured")
    except Exception as e:
        logger.warning(f"Could not setup MCP tools: {e}")

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True
)

@app.get("/ping")
def ping():
    return {"status": "healthy"}

@app.on_event("startup")
async def startup_event():
    """Initialize MCP client on startup"""
    await setup_agent_tools()

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

Writing agents/strands_aws_docs.py


#### **Optional** - Local Test

If you want to test this code locally, you can open a bash/terminal window and execute following snippets:

```bash
uv run agents/strands_aws_docs.py
```

Server will start locally. Then, run in another terminal/bash following command to test it:

```bash
curl -X POST http://0.0.0.0:9000 \-H "Content-Type: application/json" \-d '{  "jsonrpc": "2.0",  "id": "req-001",  "method": "message/send",  "params": {  "message": {  "role": "user",  "parts": [  {  "kind": "text",  "text": "What's AWS Lambda?"  }  ],  "messageId": "d0673ab9-796d-4270-9435-451912020cd1"  }  } }' | jq .
```

It will query MCP and then return an answer explaining AWS Lambda.

You can also test agent card information retrieval, using following command:

```bash
curl http://localhost:9000/.well-known/agent-card.json | jq .
```

#### 1.2 - AWS Blogs expert Agent

Now, let's write our second agent code to a local file.

In [7]:
%%writefile agents/strands_aws_blogs_news.py
import logging
import os
import asyncio
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer
import uvicorn
from fastapi import FastAPI

from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')

@tool
async def fast_internet_search(keywords: str, max_results: int = 3) -> str:
    """Fast web search with timeouts.
    Args:
        keywords (str): Search query keywords
        max_results (int): Max results (default 3 for speed)
    Returns:
        Search results
    """
    try:
        # Add AWS-specific terms for better results
        aws_keywords = f"site:aws.amazon.com {keywords} AWS"

        # Use asyncio timeout for the search
        async def search_with_timeout():
            return DDGS().text(
                aws_keywords,
                region="us-en",
                max_results=max_results
            )

        results = await asyncio.wait_for(search_with_timeout(), timeout=8.0)

        if results:
            # Format results concisely
            formatted = []
            for i, result in enumerate(results[:max_results], 1):
                formatted.append(f"{i}. {result.get('title', 'No title')}\n   {result.get('href', '')}")

            return "\n".join(formatted)
        else:
            return "No AWS results found."

    except asyncio.TimeoutError:
        logger.warning(f"Search timeout for: {keywords}")
        return "Search timed out. Try a more specific query."
    except RatelimitException:
        logger.warning("Rate limit hit")
        return "Rate limit reached. Please try again in a moment."
    except (DDGSException, Exception) as e:
        logger.error(f"Search error: {e}")
        return f"Search unavailable: {str(e)[:50]}"

system_prompt = """You are an AWS Blog Expert.

CRITICAL: Keep responses SHORT and RECENT.

Guidelines:
- Provide max 3 recent results
- Focus on official AWS blog posts only
- Use concise summaries (1-2 sentences per result)
- Include direct links when available
- Timeout searches after 8 seconds
- If search fails, acknowledge limitation

Search Strategy:
- Always include "AWS" in searches
- Focus on aws.amazon.com/blogs/ content
- Prioritize recent announcements"""

agent = Agent(
    system_prompt=system_prompt,
    tools=[fast_internet_search],
    name="AWS Blog/News Agent",
    description="An agent to search on Web latest AWS Blogs and News.",
)

host, port = "0.0.0.0", 9000

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True
)

app = FastAPI()

@app.get("/ping")
def ping():
    return {"status": "healthy"}

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

Writing agents/strands_aws_blogs_news.py


Let's write a requirements.txt file with dependencies that are needed for the agent.

In [8]:
%%writefile agents/requirements.txt
boto3==1.40.50
bedrock-agentcore==0.1.7
strands-agents[a2a]
strands-agents-tools
pyyaml
ddgs

Writing agents/requirements.txt


### 2 - Deploy to AgentCore Runtime

Now, let's deploy this solution into AgentCore Runtime.

#### 2.1 - Setup Cognito User Pool

Before deploy agents, we have to set up a Cognito User Pool, so it can validate users that are accessing our agents, or any other Idenitty provider like Okta, Microsoft Entra ID, etc.

We're going to import a helper class, that has methods to simplify few steps in our workshop. This helper class will import methods responsible to create Cognito User Pool

In [9]:
from helpers.utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = (
    setup_cognito_user_pool()
)  # You'll get your bearer token from this output cell.
print("Cognito setup completed ✓")

Setting up Amazon Cognito user pool...
Pool id: us-west-2_nfIB2q9S8
Discovery URL: https://cognito-idp.us-west-2.amazonaws.com/us-west-2_nfIB2q9S8/.well-known/openid-configuration
Client ID: 7rbhldrtfbbjkq1d4bcddn53sk
Bearer Token: eyJraWQiOiJOZTFSXC9WOThBeTBtdG8rbDExUzNpR0w1YzZlU3dzRlJKQ0NaWFRzN0J2bz0iLCJhbGciOiJSUzI1NiJ9.eyJzdWIiOiJiODAxNDNiMC1hMGExLTcwNDctZGM4Ny05YzI4OTBjNjI1ZDQiLCJpc3MiOiJodHRwczpcL1wvY29nbml0by1pZHAudXMtd2VzdC0yLmFtYXpvbmF3cy5jb21cL3VzLXdlc3QtMl9uZklCMnE5UzgiLCJjbGllbnRfaWQiOiI3cmJobGRydGZiYmprcTFkNGJjZGRuNTNzayIsIm9yaWdpbl9qdGkiOiIwNzk3ZDRlMi0yMmE2LTRhMDktYTlkNy0yMjcwMmNmZDY5YmUiLCJldmVudF9pZCI6ImQ4YWMzNmI5LTQ4OTItNGM1Ny1iZTFjLTZhZTQ5ZWIwZDRlMyIsInRva2VuX3VzZSI6ImFjY2VzcyIsInNjb3BlIjoiYXdzLmNvZ25pdG8uc2lnbmluLnVzZXIuYWRtaW4iLCJhdXRoX3RpbWUiOjE3NzU0NDczMDUsImV4cCI6MTc3NTQ1MDkwNSwiaWF0IjoxNzc1NDQ3MzA1LCJqdGkiOiJlNDQxMzQ2ZS0yODdmLTRhNDYtYWViNS1jZTM5ODI3YmQzMGUiLCJ1c2VybmFtZSI6InRlc3R1c2VyIn0.RzhJ0aQbStce-NHu-ZhcqHYLXRM2mg2yWSXq8z8DWxsBY0GAVeoBHGENgZ406Ovgsl6-A_KIc6J

#### 2.2 - Create IAM Role for the Agents

##### 2.2.1 AWS Docs Agent Execution Role

In [10]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_DOCS_ROLE_NAME

execution_role_arn_mcp = create_agentcore_runtime_execution_role(AWS_DOCS_ROLE_NAME)

✅ Created IAM role: AWSDocsAssistantBedrockAgentCoreRole-us-west-2
Role ARN: arn:aws:iam::372080370602:role/AWSDocsAssistantBedrockAgentCoreRole-us-west-2
✅ Created policy: AWSDocsAssistantBedrockAgentCorePolicy-us-west-2
✅ Attached policy to role
Policy ARN: arn:aws:iam::372080370602:policy/AWSDocsAssistantBedrockAgentCorePolicy-us-west-2


##### 2.2.2 AWS Blogs Agent Execution Role

In [11]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_BLOG_ROLE_NAME

execution_role_arn_blogs = create_agentcore_runtime_execution_role(AWS_BLOG_ROLE_NAME)

✅ Created IAM role: AWSBlogsAssistantBedrockAgentCoreRole-us-west-2
Role ARN: arn:aws:iam::372080370602:role/AWSBlogsAssistantBedrockAgentCoreRole-us-west-2
ℹ️ Policy AWSDocsAssistantBedrockAgentCorePolicy-us-west-2 already exists
✅ Attached policy to role
Policy ARN: arn:aws:iam::372080370602:policy/AWSDocsAssistantBedrockAgentCorePolicy-us-west-2


##### Create configurations for deployment on AgentCore Runtime

In following section, we're taking advantage of [starter toolkit](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-starter-toolkit.html). The starter toolkit is a Command Line Interface (CLI) toolkit that you can use to deploy AI agents to an AgentCore Runtime.

We will now create an agent with support for A2A protocol inside AgentCore runtime.

##### 2.2.3 - Let's configure and deploy our first agent (AWS Docs Agent):

In [17]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_mcp_agent = Runtime()
aws_docs_agent_name="aws_docs_assistant_" + user_name

region = boto_session.region_name

# Configure the deployment
response_aws_docs_agent = agentcore_runtime_mcp_agent.configure(
    entrypoint="agents/strands_aws_docs.py",
    execution_role=execution_role_arn_mcp,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_docs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A",
)

print("Configuration completed:", response_aws_docs_agent)

Entrypoint parsed: file=/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/agents/strands_aws_docs.py, bedrock_agentcore_name=strands_aws_docs
Memory configured with STM only
Configuring BedrockAgentCore agent: aws_docs_assistant_eric_fu
Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/.dockerignore
Keeping 'aws_docs_assistant_eric_fu' as default agent
Bedrock AgentCore configured: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/.bedrock_agentcore.yaml


Configuration completed: config_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/Dockerfile') dockerignore_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/.dockerignore') runtime='Docker' region='us-west-2' account_id='372080370602' execution_role='arn:aws:iam::372080370602:role/AWSDocsAssistantBedrockAgentCoreRole-us-west-2' ecr_repository=None auto_create_ecr=True memory_id=None


Launch the first agent on AgentCore Runtime

In [18]:
launch_result_mcp = agentcore_runtime_mcp_agent.launch()
print("Launch completed:", launch_result_mcp.agent_arn)

docs_agent_arn = launch_result_mcp.agent_arn

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Creating memory resource for agent: aws_docs_assistant_eric_fu
✅ MemoryManager initialized for region: us-west-2
Creating new STM-only memory...
Created memory: aws_docs_assistant_eric_fu_mem-0NjSKHC9KF
Memory created but flag was False - correcting to True
✅ New memory created: aws_docs_assistant_eric_fu_mem-0NjSKHC9KF (provisioning in background)
Starting CodeBuild ARM64 deployment for agent 'aws_docs_assistant_eric_fu' to account 372080370602 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: aws_docs_assistant_eric_fu
✅ ECR 

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-aws_docs_assistant_eric_fu


Using execution role from config: arn:aws:iam::372080370602:role/AWSDocsAssistantBedrockAgentCoreRole-us-west-2
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: aws_docs_assistant_eric_fu
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-06c37326ba
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-06c37326ba
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-06c37326ba
✓ Role created: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-06c37326ba
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-06c37326ba
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-06c37326ba
Using dockerignore.template with 45 patterns for zip filtering
Uploaded sourc

Launch completed: arn:aws:bedrock-agentcore:us-west-2:372080370602:runtime/aws_docs_assistant_eric_fu-X7QG3rFSOq


**Check Deployment Status**

Let's check if deployment is completed:

In [19]:
status_response = agentcore_runtime_mcp_agent.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: aws_docs_assistant_eric_fu_mem-0NjSKHC9KF...
  Found memory: aws_docs_assistant_eric_fu_mem-0NjSKHC9KF
Retrieved Bedrock AgentCore status for: aws_docs_assistant_eric_fu


Final status: READY


##### 2.2.4 - Let's configure and deploy our second agent (AWS Blogs and News Agent):

In [20]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_blogs = Runtime()
aws_blogs_agent_name="aws_blog_assistant_" + user_name

# Configure the deployment
response_aws_blogs_agent = agentcore_runtime_blogs.configure(
    entrypoint="agents/strands_aws_blogs_news.py",
    execution_role=execution_role_arn_blogs,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_blogs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A"
)

print("Configuration completed:", response_aws_blogs_agent)

Entrypoint parsed: file=/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/agents/strands_aws_blogs_news.py, bedrock_agentcore_name=strands_aws_blogs_news
Memory configured with STM only
Configuring BedrockAgentCore agent: aws_blog_assistant_eric_fu
Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/.dockerignore
Changing default agent from 'aws_docs_assistant_eric_fu' to 'aws_blog_assistant_eric_fu'
Bedrock AgentCore configured: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/.bedrock_agentcore.yaml


Configuration completed: config_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/Dockerfile') dockerignore_path=PosixPath('/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/05-hosting-a2a/01-a2a-agent-orchestrating-mcp/.dockerignore') runtime='Docker' region='us-west-2' account_id='372080370602' execution_role='arn:aws:iam::372080370602:role/AWSBlogsAssistantBedrockAgentCoreRole-us-west-2' ecr_repository=None auto_create_ecr=True memory_id=None


Launch the second agent on AgentCore Runtime

In [21]:
launch_result_blog = agentcore_runtime_blogs.launch()
print("Launch completed:", launch_result_blog.agent_arn)

blog_agent_arn = launch_result_blog.agent_arn

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Creating memory resource for agent: aws_blog_assistant_eric_fu
✅ MemoryManager initialized for region: us-west-2
Creating new STM-only memory...
Created memory: aws_blog_assistant_eric_fu_mem-i6gob9EtlN
Memory created but flag was False - correcting to True
✅ New memory created: aws_blog_assistant_eric_fu_mem-i6gob9EtlN (provisioning in background)
Starting CodeBuild ARM64 deployment for agent 'aws_blog_assistant_eric_fu' to account 372080370602 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: aws_blog_assistant_eric_fu
✅ ECR 

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-aws_blog_assistant_eric_fu


Using execution role from config: arn:aws:iam::372080370602:role/AWSBlogsAssistantBedrockAgentCoreRole-us-west-2
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: aws_blog_assistant_eric_fu
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-95420316ba
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-95420316ba
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-95420316ba
✓ Role created: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-95420316ba
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-95420316ba
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-95420316ba
Using dockerignore.template with 45 patterns for zip filtering
Uploaded sour

Launch completed: arn:aws:bedrock-agentcore:us-west-2:372080370602:runtime/aws_blog_assistant_eric_fu-wm2fy17gHR


**Check Deployment Status**

Let's check if deployment of second agent is completed:

In [22]:
status_response = agentcore_runtime_blogs.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: aws_blog_assistant_eric_fu_mem-i6gob9EtlN...
  Found memory: aws_blog_assistant_eric_fu_mem-i6gob9EtlN
Retrieved Bedrock AgentCore status for: aws_blog_assistant_eric_fu


Final status: READY


##### 2.2.5 - Export and save outputs

Export variables to be used in next notebooks:

In [23]:
MCP_AGENT_ID = launch_result_mcp.agent_id
MCP_AGENT_ARN = launch_result_mcp.agent_arn
MCP_AGENT_NAME = aws_docs_agent_name

BLOG_AGENT_ID = launch_result_blog.agent_id
BLOG_AGENT_ARN = launch_result_blog.agent_arn
BLOG_AGENT_NAME = aws_blogs_agent_name

COGNITO_CLIENT_ID = cognito_config.get("client_id")
COGNITO_SECRET = cognito_config.get("client_secret")
DISCOVERY_URL = cognito_config.get("discovery_url")

%store MCP_AGENT_ID
%store MCP_AGENT_ARN
%store MCP_AGENT_NAME
%store BLOG_AGENT_ID
%store BLOG_AGENT_ARN
%store BLOG_AGENT_NAME
%store COGNITO_CLIENT_ID
%store COGNITO_SECRET
%store DISCOVERY_URL

Stored 'MCP_AGENT_ID' (str)
Stored 'MCP_AGENT_ARN' (str)
Stored 'MCP_AGENT_NAME' (str)
Stored 'BLOG_AGENT_ID' (str)
Stored 'BLOG_AGENT_ARN' (str)
Stored 'BLOG_AGENT_NAME' (str)
Stored 'COGNITO_CLIENT_ID' (str)
Stored 'COGNITO_SECRET' (str)
Stored 'DISCOVERY_URL' (str)


Store ARN of the agents in SSM, so it can be used by orchestrator:

In [24]:
from helpers.utils import put_ssm_parameter, SSM_DOCS_AGENT_ARN, SSM_BLOGS_AGENT_ARN

put_ssm_parameter(SSM_DOCS_AGENT_ARN, MCP_AGENT_ARN)

put_ssm_parameter(SSM_BLOGS_AGENT_ARN, BLOG_AGENT_ARN)

### 3 - Invoking A2A agents

Firstly, let's refresh the auth token:

In [25]:
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"),
    cognito_config.get("client_secret")
)

#### 3.1 Getting Agent Cards

Now let's get started by getting the Agent Card information from our first agent (AWS Docs MCP Expert):

In [26]:
import logging
from uuid import uuid4
from urllib.parse import quote

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

def fetch_agent_card(agent_arn):
    # URL encode the agent ARN
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/.well-known/agent-card.json"
    logger.info(url)
    # Generate a unique session ID
    session_id = str(uuid4())
    logger.info(f"Generated session ID: {session_id}")

    # Set headers
    headers = {
        'Accept': '*/*',
        'Authorization': f'Bearer {bearer_token}',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
        'X-Amzn-Trace-Id': f'aws_docs_assistant_{session_id}'
    }

    try:
        # Make the request
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parse and pretty print JSON
        agent_card = response.json()
        logger.info(json.dumps(agent_card, indent=2))

        return agent_card

    except requests.exceptions.RequestException as e:
        logger.error(f"Error fetching agent card: {e}")
        return None

In [27]:
fetch_agent_card(docs_agent_arn)

{'capabilities': {'streaming': True},
 'defaultInputModes': ['text'],
 'defaultOutputModes': ['text'],
 'description': 'An agent to query AWS Docs using AWS MCP.',
 'name': 'AWS Docs Agent',
 'preferredTransport': 'JSONRPC',
 'protocolVersion': '0.3.0',
 'skills': [],
 'url': 'https://bedrock-agentcore.us-west-2.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-west-2%3A372080370602%3Aruntime%2Faws_docs_assistant_eric_fu-X7QG3rFSOq/invocations/',
 'version': '0.0.1'}

Not let's check the agent card for the second agent (AWS Blogs and News expert): 

In [28]:
fetch_agent_card(blog_agent_arn)

{'capabilities': {'streaming': True},
 'defaultInputModes': ['text'],
 'defaultOutputModes': ['text'],
 'description': 'An agent to search on Web latest AWS Blogs and News.',
 'name': 'AWS Blog/News Agent',
 'preferredTransport': 'JSONRPC',
 'protocolVersion': '0.3.0',
 'skills': [{'description': 'Fast web search with timeouts.\nReturns:\n    Search results',
   'id': 'fast_internet_search',
   'name': 'fast_internet_search',
   'tags': []}],
 'url': 'https://bedrock-agentcore.us-west-2.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-west-2%3A372080370602%3Aruntime%2Faws_blog_assistant_eric_fu-wm2fy17gHR/invocations/',
 'version': '0.0.1'}

#### 3.2 - Test agents

Now, let's invoke the first agent, using A2A:

In [29]:
import asyncio
import logging
import os
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 300  # set request timeout to 5 minutes

def format_agent_response(response):
    """Extract and format agent response for human readability."""
    # Get the main response text from artifacts
    if response.artifacts and len(response.artifacts) > 0:
        artifact = response.artifacts[0]
        if artifact.parts and len(artifact.parts) > 0:
            return artifact.parts[0].root.text

    # Fallback: concatenate all agent messages from history
    agent_messages = [
        msg.parts[0].root.text
        for msg in response.history
        if msg.role.value == 'agent' and msg.parts
    ]
    return ''.join(agent_messages)


def create_message(*, role: Role = Role.user, text: str) -> Message:
    return Message(
        kind="message",
        role=role,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )

async def send_sync_message(agent_arn, message: str):
    # Get runtime URL from environment variable
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/"

    # Generate a unique session ID
    session_id = str(uuid4())
    print(f"Generated session ID: {session_id}")

    # Add authentication headers for AgentCore
    headers = {"Authorization": f"Bearer {bearer_token}",
              'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id}

    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, headers=headers) as httpx_client:
        # Get agent card from the runtime URL
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
        agent_card = await resolver.get_agent_card()
        print(agent_card)

        # Agent card contains the correct URL (same as runtime_url in this case)
        # No manual override needed - this is the path-based mounting pattern

        # Create client using factory
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=False,  # Use non-streaming mode for sync response
        )
        factory = ClientFactory(config)
        client = factory.create(agent_card)

        # Create and send message
        msg = create_message(text=message)

        # With streaming=False, this will yield exactly one result
        async for event in client.send_message(msg):
            if isinstance(event, Message):
                logger.info(event.model_dump_json(exclude_none=True, indent=2))
                return event
            elif isinstance(event, tuple) and len(event) == 2:
                # (Task, UpdateEvent) tuple
                task, update_event = event
                logger.info(f"Task: {task.model_dump_json(exclude_none=True, indent=2)}")
                if update_event:
                    logger.info(f"Update: {update_event.model_dump_json(exclude_none=True, indent=2)}")
                return task
            else:
                # Fallback for other response types
                logger.info(f"Response: {str(event)}")
                return event

In [30]:
result = await send_sync_message(docs_agent_arn, "what is DynamoDB")
formatted_output = format_agent_response(result)
print(formatted_output)

Generated session ID: 6d2bb08e-77dd-43e3-9c33-ee0d9db27a2b
additional_interfaces=None capabilities=AgentCapabilities(extensions=None, push_notifications=None, state_transition_history=None, streaming=True) default_input_modes=['text'] default_output_modes=['text'] description='An agent to query AWS Docs using AWS MCP.' documentation_url=None icon_url=None name='AWS Docs Agent' preferred_transport='JSONRPC' protocol_version='0.3.0' provider=None security=None security_schemes=None signatures=None skills=[] supports_authenticated_extended_card=None url='https://bedrock-agentcore.us-west-2.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-west-2%3A372080370602%3Aruntime%2Faws_docs_assistant_eric_fu-X7QG3rFSOq/invocations/' version='0.0.1'
Amazon DynamoDB is a fully managed NoSQL database service that provides fast and predictable performance with seamless scalability. It supports key-value and document data models, making it ideal for mobile, web, gaming, ad-tech, IoT, and other a

Not, let's test our 2nd agent:

In [31]:
result = await send_sync_message(blog_agent_arn, "Give me the latest published blog for Bedrock AgentCore?")
formatted_output = format_agent_response(result)
print(formatted_output)

Generated session ID: c10bcab8-6155-4eb2-b2c5-b09fd186b51c
additional_interfaces=None capabilities=AgentCapabilities(extensions=None, push_notifications=None, state_transition_history=None, streaming=True) default_input_modes=['text'] default_output_modes=['text'] description='An agent to search on Web latest AWS Blogs and News.' documentation_url=None icon_url=None name='AWS Blog/News Agent' preferred_transport='JSONRPC' protocol_version='0.3.0' provider=None security=None security_schemes=None signatures=None skills=[AgentSkill(description='Fast web search with timeouts.\nReturns:\n    Search results', examples=None, id='fast_internet_search', input_modes=None, name='fast_internet_search', output_modes=None, security=None, tags=[])] supports_authenticated_extended_card=None url='https://bedrock-agentcore.us-west-2.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-west-2%3A372080370602%3Aruntime%2Faws_blog_assistant_eric_fu-wm2fy17gHR/invocations/' version='0.0.1'
Here are the

Following is a more detailed output, showing steps that agent has taken.

Feel free to change the questions asked to agent and see the step-by-step result.

In [32]:
def format_agent_trace(response):
    """Format agent response as a readable trace of calls."""
    print("=" * 60)
    print("🔍 AGENT EXECUTION TRACE")
    print("=" * 60)

    # Context info
    print(f"📋 Context ID: {response.context_id}")
    print(f"🆔 Task ID: {response.id}")
    print(f"📊 Status: {response.status.state.value}")
    print(f"⏰ Completed: {response.status.timestamp}")
    print()

    # Trace through history
    print("🔄 EXECUTION FLOW:")
    print("-" * 40)

    for i, msg in enumerate(response.history, 1):
        role_icon = "👤" if msg.role.value == "user" else "🤖"
        text = msg.parts[0].root.text if msg.parts else "[No content]"

        # Truncate long messages for trace view
        if len(text) > 80:
            text = text[:77] + "..."

        print(f"{i:2d}. {role_icon} {msg.role.value.upper()}: {text}")

    print()
    print("✅ FINAL RESULT:")
    print("-" * 40)

    # Final artifact
    if response.artifacts:
        final_text = response.artifacts[0].parts[0].root.text
        print(final_text[:200] + "..." if len(final_text) > 200 else final_text)

    print("=" * 60)

In [33]:
format_agent_trace(result)

🔍 AGENT EXECUTION TRACE
📋 Context ID: 786db1df-2b08-4c83-b964-a13266ff93b1
🆔 Task ID: fa73183c-3c83-4a03-a68f-2b7522cffc39
📊 Status: completed
⏰ Completed: 2026-04-06T04:03:36.884978+00:00

🔄 EXECUTION FLOW:
----------------------------------------
 1. 👤 USER: Give me the latest published blog for Bedrock AgentCore?
 2. 🤖 AGENT: Here
 3. 🤖 AGENT:  are the latest AWS
 4. 🤖 AGENT:  blog posts for Bedrock Agent
 5. 🤖 AGENT: Core:

1. **Amazon
 6. 🤖 AGENT:  Bedrock AgentCore adds
 7. 🤖 AGENT:  quality evaluations and policy controls for deplo
 8. 🤖 AGENT: ying trusted AI agents**
 9. 🤖 AGENT:  - Latest
10. 🤖 AGENT:  update
11. 🤖 AGENT:  covering
12. 🤖 AGENT:  new
13. 🤖 AGENT:  evaluation
14. 🤖 AGENT:  capabilities
15. 🤖 AGENT:  and governance
16. 🤖 AGENT:  controls
17. 🤖 AGENT:  for
18. 🤖 AGENT:  AI agent
19. 🤖 AGENT:  deployment.
20. 🤖 AGENT: 
   [
21. 🤖 AGENT: Rea
22. 🤖 AGENT: d more
23. 🤖 AGENT: ](https://aws.amazon.com
24. 🤖 AGENT: /blogs/aws/amazon-be
25. 🤖 AGENT: drock-agentcore-adds

Congratulations, you have deployed your first agent, using A2A protocol on Amazon AgentCore Runtime!

Now, let's move to next lab.